In [ ]:
# --- Sekcja 1: Konfiguracja i Importy ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import sys
import os

# Dodanie ścieżki do konfiguracji projektu
sys.path.append('../src')
import config

# Ustawienia estetyczne wykresów
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

In [ ]:
# --- Sekcja 2: Funkcja Automatycznego Raportu EDA ---
def perform_medical_eda(dataset_name):
    print(f"{'='*30}\nANALIZA: {dataset_name.upper()}\n{'='*30}")
    
    # Wczytanie danych
    file_path = config.PROCESSED_DATA_DIR / f"{dataset_name}_processed.csv"
    if not file_path.exists():
        print(f"Błąd: Plik {file_path} nie istnieje. Uruchom najpierw preprocessor.py.")
        return
    
    df = pd.read_csv(file_path)
    
    # Pobranie metadanych ze słownika (z config.py)
    if dataset_name in config.UCI_DATASETS:
        target_col = config.UCI_DATASETS[dataset_name]['target_col']
    else:
        target_col = config.OPENML_DATASETS[dataset_name]['target_col']

    # 1. Analiza brakujących danych (Missingness Map)
    # Kluczowe dla zbioru Chronic Kidney Disease (CKD)
    plt.figure(figsize=(10, 4))
    msno.matrix(df)
    plt.title(f"Mapa brakujących danych: {dataset_name}")
    plt.show()

    # 2. Rozkład klasy docelowej (Class Imbalance)
    # Krytyczne dla zbioru Cervical Cancer
    plt.figure(figsize=(6, 4))
    sns.countplot(x=target_col, data=df, palette='viridis')
    plt.title(f"Rozkład klas (Target: {target_col})")
    plt.show()
    
    imbalance_ratio = df[target_col].value_counts(normalize=True).min()
    print(f"Najmniejsza klasa stanowi: {imbalance_ratio:.2%} zbioru.")

    # 3. Macierz Korelacji (Redundancja Cech)
    # Ważne dla Parkinson's (wiele cech akustycznych może być silnie skorelowanych)
    plt.figure(figsize=(12, 10))
    corr = df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='RdBu', center=0, annot=False, linewidths=.5)
    plt.title(f"Korelacje cech klinicznych: {dataset_name}")
    plt.show()

    # 4. Statystyki opisowe
    print("\nStatystyki kluczowych cech:")
    display(df.describe().T.head(10)) # Pokazujemy pierwsze 10 cech dla czytelności

In [ ]:
# --- Sekcja 3: Uruchomienie dla wszystkich zbiorów ---
datasets_to_analyze = list(config.UCI_DATASETS.keys()) + list(config.OPENML_DATASETS.keys())

for ds in datasets_to_analyze:
    perform_medical_eda(ds)